# Sparse Hyperparameter Tuning (Simplified, Colab Ready)

This notebook tunes **core sparse hyperparameters only**:
- `window_size`
- `expander_degree`
- learning rate (`lr`)

Norm control is fixed to the winner from norm tuning:
- `qk_norm=False`, `ortho_init=False`, `spectral_norm=False`

For norm-control sweeps, use `sparse_norm_control_tuning.ipynb`.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_REF = os.environ.get("ORION_GIT_REF", "main")  # set to branch/tag/sha if needed

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/orion")
    REPO_ROOT = DRIVE_ROOT / "repo"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()


def _run(
    cmd: list[str], *, cwd: Path | None = None, check: bool = True
) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({' '.join(cmd)}):\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists()


def _clone_repo(repo_url: str, repo_root: Path, retries: int = 3) -> None:
    last_error = "unknown"
    for attempt in range(1, retries + 1):
        proc = subprocess.run(
            ["git", "clone", repo_url, str(repo_root)],
            text=True,
            capture_output=True,
        )
        if proc.returncode == 0:
            return

        last_error = (proc.stderr or proc.stdout or "clone failed").strip()
        tail = last_error.splitlines()[-1] if last_error else "clone failed"
        print(f"Clone attempt {attempt}/{retries} failed: {tail}")

        if repo_root.exists() and not _is_git_repo(repo_root):
            shutil.rmtree(repo_root, ignore_errors=True)

        if attempt < retries:
            time.sleep(attempt)

    raise RuntimeError(f"Failed to clone repository after {retries} attempts: {last_error}")


def _sync_repo(repo_root: Path, ref: str) -> None:
    # Fail fast on dirty trees so runs stay reproducible.
    status = _run(["git", "status", "--porcelain"], cwd=repo_root)
    if status.stdout.strip():
        raise RuntimeError(
            "Repository has local changes in Drive clone. "
            "Commit/stash/remove them, or delete the repo folder and rerun."
        )

    _run(["git", "fetch", "--all", "--tags", "--prune"], cwd=repo_root)

    # Checkout requested ref; works for local branch, remote branch, tag, or sha.
    checkout = _run(["git", "checkout", ref], cwd=repo_root, check=False)
    if checkout.returncode != 0:
        # Try remote branch fallback.
        _run(["git", "checkout", "-B", ref, f"origin/{ref}"], cwd=repo_root)

    # Fast-forward if branch tracks remote.
    branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_root).stdout.strip()
    if branch != "HEAD":
        _run(["git", "pull", "--ff-only", "origin", branch], cwd=repo_root)


if REPO_ROOT.exists() and not _is_git_repo(REPO_ROOT):
    if any(REPO_ROOT.iterdir()):
        backup = REPO_ROOT.with_name(f"{REPO_ROOT.name}_backup_{int(time.time())}")
        print(f"Found non-git directory at {REPO_ROOT}; moving to {backup}")
        REPO_ROOT.rename(backup)
    else:
        REPO_ROOT.rmdir()

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    _clone_repo(REPO_URL, REPO_ROOT)
else:
    print(f"Using existing repository at {REPO_ROOT}")

_sync_repo(REPO_ROOT, REPO_REF)
os.chdir(REPO_ROOT)

commit = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT).stdout.strip()
branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT).stdout.strip()

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"Checked out: {branch} @ {commit}")
if IN_COLAB and USE_GOOGLE_DRIVE:
    print(f"Google Drive persistence enabled at: {REPO_ROOT}")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn", "matplotlib"], check=True
)
print("Dependencies installed.")

In [ ]:
import itertools
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

from orion.data.shakespeare import load_tiny_shakespeare
from orion.experiments import evaluate_val_ppl

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
# =============================
# Tuning settings (edit these)
# =============================
EXPERIMENT_NAME = "sparse_hparam_tuning_simple"

SEQ_LEN = 1024
BATCH_BY_SEQ = {256: 8, 512: 4, 1024: 2, 2048: 1, 4096: 1}

WINDOW_GRID = [64, 128]
EXPANDER_DEGREE_GRID = [8, 16, 32, 64, 128, 256]
LR_GRID = [3e-4, 2e-4]

# Norm-control winner fixed here
FIXED_STABILITY = {"qk_norm": False, "ortho_init": False, "spectral_norm": False}

STAGE1_SEEDS = [123]
STAGE2_SEEDS = [123, 456]
STAGE1_TRAIN_TOKENS = 1_500_000
STAGE2_TRAIN_TOKENS = 4_000_000
TOP_K = 4
VAL_EVAL_BATCHES_STAGE2 = 40
MAX_STAGE1_RUNS = None  # set int for quick debug

MODEL_CFG = {
    "name": "orion",
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "mlp_mult": 4,
}

assert SEQ_LEN in BATCH_BY_SEQ, f"Missing batch mapping for seq_len={SEQ_LEN}"
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required for sparse_impl='flex'. Enable GPU runtime in Colab.")

DEVICE = "cuda"
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("runs") / f"{EXPERIMENT_NAME}_{TIMESTAMP}"
CFG_ROOT = RUN_ROOT / "generated_configs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CFG_ROOT.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Run root:", RUN_ROOT)
print("Fixed stability:", FIXED_STABILITY)

load_tiny_shakespeare("data")
print("Dataset ready.")

In [ ]:
def steps_for_token_budget(seq_len: int, batch_size: int, token_budget: int) -> int:
    return max(1, math.ceil(token_budget / (seq_len * batch_size)))


def build_sparse_cfg(
    *,
    out_dir: Path,
    seed: int,
    seq_len: int,
    batch_size: int,
    lr: float,
    window_size: int,
    expander_degree: int,
    train_tokens_target: int,
) -> dict:
    steps = steps_for_token_budget(seq_len, batch_size, train_tokens_target)
    return {
        "run": {
            "out_dir": str(out_dir),
            "seed": int(seed),
            "steps": int(steps),
            "log_every": 10,
            "save_every": int(steps),
        },
        "data": {
            "dataset": "tinyshakespeare",
            "root": "data",
            "seq_len": int(seq_len),
            "batch_size": int(batch_size),
        },
        "model": MODEL_CFG.copy(),
        "attention": {
            "backend": "sparse",
            "window_size": int(window_size),
            "expander_degree": int(expander_degree),
            "sparse_impl": "flex",
            "sparse_block_size": 128,
            "sparse_probe_every": 50,
            "sparse_probe_tokens": 256,
        },
        "stability": FIXED_STABILITY.copy(),
        "optim": {"lr": float(lr)},
        "eval": {"long_context_batch_size": 1},
    }


def write_yaml(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")


def run_train_cfg(cfg_path: Path) -> tuple[int, str, str, float]:
    cmd = [sys.executable, "-m", "orion.train", "--config", str(cfg_path), "--device", DEVICE]
    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    t0 = time.time()
    proc = subprocess.run(cmd, text=True, capture_output=True, env=env, cwd=Path.cwd())
    dt = time.time() - t0
    return proc.returncode, proc.stdout, proc.stderr, dt


def read_metrics(metrics_path: Path) -> dict:
    if not metrics_path.exists():
        return {}

    step_rows = []
    window_rows = []
    with metrics_path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            t = row.get("type")
            if t == "step":
                step_rows.append(row)
            elif t == "window":
                window_rows.append(row)

    out = {}
    if step_rows:
        tail = step_rows[-min(20, len(step_rows)) :]
        out["final_loss"] = float(step_rows[-1].get("loss", math.nan))
        out["final_ppl"] = float(step_rows[-1].get("ppl", math.nan))
        out["train_tok_per_s"] = float(
            np.mean([r.get("throughput_tokens_per_sec", math.nan) for r in tail])
        )

    if window_rows:
        w = window_rows[-1]
        out["vram_peak_mib"] = float(w.get("vram_peak_mib", math.nan))
        out["valid_vs_causal_cap"] = float(w.get("valid_neighbor_fraction_vs_causal_cap", math.nan))

    return out


def minmax_score(series: pd.Series, higher_is_better: bool) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return pd.Series(np.ones(len(series)), index=series.index)
    norm = (s - lo) / (hi - lo)
    return norm if higher_is_better else (1.0 - norm)

In [ ]:
search_df = (
    pd.DataFrame(
        [
            {"window_size": w, "expander_degree": d, "lr": lr}
            for w, d, lr in itertools.product(WINDOW_GRID, EXPANDER_DEGREE_GRID, LR_GRID)
            if w < SEQ_LEN
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

if MAX_STAGE1_RUNS is not None:
    search_df = search_df.head(int(MAX_STAGE1_RUNS)).copy()

print(f"Stage-1 candidates: {len(search_df)}")
display(search_df.head(20))

In [ ]:
stage1_rows = []
batch_size = BATCH_BY_SEQ[SEQ_LEN]

for i, cand in search_df.iterrows():
    for seed in STAGE1_SEEDS:
        trial_tag = f"s1_idx{i:03d}_T{SEQ_LEN}_w{int(cand.window_size)}_d{int(cand.expander_degree)}_lr{cand.lr}".replace(
            ".", "p"
        )
        run_dir = RUN_ROOT / "stage1" / trial_tag
        cfg_path = CFG_ROOT / "stage1" / f"{trial_tag}.yaml"

        cfg = build_sparse_cfg(
            out_dir=run_dir,
            seed=int(seed),
            seq_len=SEQ_LEN,
            batch_size=batch_size,
            lr=float(cand.lr),
            window_size=int(cand.window_size),
            expander_degree=int(cand.expander_degree),
            train_tokens_target=STAGE1_TRAIN_TOKENS,
        )
        write_yaml(cfg_path, cfg)

        print(f"[Stage1 {i + 1}/{len(search_df)}] {trial_tag}")
        rc, stdout, stderr, elapsed_s = run_train_cfg(cfg_path)

        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "stdout.log").write_text(stdout, encoding="utf-8")
        (run_dir / "stderr.log").write_text(stderr, encoding="utf-8")

        metrics = read_metrics(run_dir / "metrics.jsonl")
        err_tail = ((stderr or "") + "\n" + (stdout or ""))[-12000:]

        stage1_rows.append(
            {
                "stage": "stage1",
                "trial_tag": trial_tag,
                "status": "ok" if rc == 0 else "failed",
                "returncode": int(rc),
                "elapsed_s": float(elapsed_s),
                "seed": int(seed),
                "seq_len": int(SEQ_LEN),
                "batch_size": int(batch_size),
                "train_tokens_target": int(STAGE1_TRAIN_TOKENS),
                **cand.to_dict(),
                **metrics,
                "error_tail": err_tail if rc != 0 else "",
                "run_dir": str(run_dir),
                "cfg_path": str(cfg_path),
            }
        )

stage1_df = pd.DataFrame(stage1_rows)
stage1_df.to_csv(RUN_ROOT / "stage1_results.csv", index=False)
print(stage1_df.status.value_counts(dropna=False).to_string())
display(stage1_df.head(20))

In [ ]:
stage1_ok = stage1_df[stage1_df.status == "ok"].copy()
if len(stage1_ok) == 0:
    raise RuntimeError("No successful stage-1 runs.")

if "valid_vs_causal_cap" in stage1_ok.columns:
    stage1_ok = stage1_ok[
        (stage1_ok.valid_vs_causal_cap.isna()) | (stage1_ok.valid_vs_causal_cap >= 0.95)
    ].copy()

agg1 = stage1_ok.groupby(["window_size", "expander_degree", "lr"], as_index=False).agg(
    final_ppl_mean=("final_ppl", "mean"),
    train_tok_per_s_mean=("train_tok_per_s", "mean"),
    vram_peak_mib_mean=("vram_peak_mib", "mean"),
    runs=("trial_tag", "count"),
)

agg1["score_quality"] = minmax_score(agg1["final_ppl_mean"], higher_is_better=False)
agg1["score_speed"] = minmax_score(agg1["train_tok_per_s_mean"], higher_is_better=True)
agg1["score_vram"] = minmax_score(agg1["vram_peak_mib_mean"], higher_is_better=False)
agg1["stage1_score"] = (
    0.65 * agg1["score_quality"] + 0.25 * agg1["score_speed"] + 0.10 * agg1["score_vram"]
)
agg1 = agg1.sort_values("stage1_score", ascending=False).reset_index(drop=True)

top_candidates = agg1.head(min(TOP_K, len(agg1))).copy()
top_candidates.to_csv(RUN_ROOT / "stage1_top_candidates.csv", index=False)
print("Top stage-1 candidates:")
display(top_candidates)

In [ ]:
stage2_rows = []
batch_size = BATCH_BY_SEQ[SEQ_LEN]

for rank, cand in top_candidates.iterrows():
    for seed in STAGE2_SEEDS:
        trial_tag = f"s2_rank{rank + 1:02d}_T{SEQ_LEN}_w{int(cand.window_size)}_d{int(cand.expander_degree)}_lr{cand.lr}_seed{seed}".replace(
            ".", "p"
        )
        run_dir = RUN_ROOT / "stage2" / trial_tag
        cfg_path = CFG_ROOT / "stage2" / f"{trial_tag}.yaml"

        cfg = build_sparse_cfg(
            out_dir=run_dir,
            seed=int(seed),
            seq_len=SEQ_LEN,
            batch_size=batch_size,
            lr=float(cand.lr),
            window_size=int(cand.window_size),
            expander_degree=int(cand.expander_degree),
            train_tokens_target=STAGE2_TRAIN_TOKENS,
        )
        write_yaml(cfg_path, cfg)

        print(f"[Stage2 {rank + 1}/{len(top_candidates)}] {trial_tag}")
        rc, stdout, stderr, elapsed_s = run_train_cfg(cfg_path)

        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "stdout.log").write_text(stdout, encoding="utf-8")
        (run_dir / "stderr.log").write_text(stderr, encoding="utf-8")

        metrics = read_metrics(run_dir / "metrics.jsonl")

        val_loss = math.nan
        val_ppl = math.nan
        val_error = ""
        if rc == 0 and (run_dir / "checkpoint.pt").exists():
            try:
                val_loss, val_ppl = evaluate_val_ppl(
                    cfg, run_dir / "checkpoint.pt", batches=VAL_EVAL_BATCHES_STAGE2
                )
            except Exception as exc:
                val_error = str(exc)

        err_tail = ((stderr or "") + "\n" + (stdout or ""))[-12000:]
        stage2_rows.append(
            {
                "stage": "stage2",
                "trial_tag": trial_tag,
                "status": "ok" if rc == 0 else "failed",
                "returncode": int(rc),
                "elapsed_s": float(elapsed_s),
                "seed": int(seed),
                "seq_len": int(SEQ_LEN),
                "batch_size": int(batch_size),
                "train_tokens_target": int(STAGE2_TRAIN_TOKENS),
                **{
                    "window_size": cand.window_size,
                    "expander_degree": cand.expander_degree,
                    "lr": cand.lr,
                },
                **metrics,
                "val_loss": val_loss,
                "val_ppl": val_ppl,
                "val_eval_error": val_error,
                "error_tail": err_tail if rc != 0 else "",
                "run_dir": str(run_dir),
                "cfg_path": str(cfg_path),
            }
        )

stage2_df = pd.DataFrame(stage2_rows)
stage2_df.to_csv(RUN_ROOT / "stage2_results.csv", index=False)
print(stage2_df.status.value_counts(dropna=False).to_string())
display(stage2_df.head(20))

In [ ]:
stage2_ok = stage2_df[stage2_df.status == "ok"].copy()
if len(stage2_ok) == 0:
    raise RuntimeError("No successful stage-2 runs.")

summary = stage2_ok.groupby(["window_size", "expander_degree", "lr"], as_index=False).agg(
    val_ppl_mean=("val_ppl", "mean"),
    val_ppl_std=("val_ppl", "std"),
    final_ppl_mean=("final_ppl", "mean"),
    train_tok_per_s_mean=("train_tok_per_s", "mean"),
    vram_peak_mib_mean=("vram_peak_mib", "mean"),
    runs=("trial_tag", "count"),
)

quality_col = "val_ppl_mean" if summary["val_ppl_mean"].notna().any() else "final_ppl_mean"
summary["score_quality"] = minmax_score(summary[quality_col], higher_is_better=False)
summary["score_speed"] = minmax_score(summary["train_tok_per_s_mean"], higher_is_better=True)
summary["score_vram"] = minmax_score(summary["vram_peak_mib_mean"], higher_is_better=False)
summary["final_score"] = (
    0.60 * summary["score_quality"] + 0.25 * summary["score_speed"] + 0.15 * summary["score_vram"]
)
summary = summary.sort_values("final_score", ascending=False).reset_index(drop=True)
summary.to_csv(RUN_ROOT / "stage2_summary_ranked.csv", index=False)

best = summary.iloc[0].to_dict()
print("Ranked summary:")
display(summary)
print("\nBest sparse combo:")
for k in ["window_size", "expander_degree", "lr", "final_score"]:
    print(f"{k}: {best.get(k)}")

best_cfg = build_sparse_cfg(
    out_dir=RUN_ROOT / "best_sparse_final_run",
    seed=int(STAGE2_SEEDS[0]),
    seq_len=int(SEQ_LEN),
    batch_size=int(BATCH_BY_SEQ[SEQ_LEN]),
    lr=float(best["lr"]),
    window_size=int(best["window_size"]),
    expander_degree=int(best["expander_degree"]),
    train_tokens_target=int(STAGE2_TRAIN_TOKENS),
)

best_cfg_path = RUN_ROOT / "best_sparse_config.yaml"
write_yaml(best_cfg_path, best_cfg)
print("Saved best config:", best_cfg_path)

repo_variant_path = Path("configs/experiments/variants/sparse_tuned_best.yaml")
write_yaml(repo_variant_path, best_cfg)
print("Updated reusable variant:", repo_variant_path)

plt.figure(figsize=(10, 5))
sns.boxplot(data=stage2_ok, x="expander_degree", y="val_ppl")
plt.title("Validation PPL by Expander Degree (Stage 2)")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=stage2_ok, x="expander_degree", y="train_tok_per_s")
plt.title("Training Throughput by Expander Degree (Stage 2)")
plt.show()